# Two-dimensional Wave Equation

This notebook tests tensor parametric operator inference on a hyperbolic partial differential equation (PDE) in two spatial dimensions.
To begin, import a few standard Python scientific libraries, the [`opinf`](https://willcox-research-group.github.io/rom-operator-inference-Python3) package, and a few local files.

In [ ]:
import opinf
import warnings
import collections
import numpy as np
import matplotlib.pyplot as plt

import utils
import models
import waveEq as wave

In [ ]:
utils.matplotlib_config()

If any of these imports fail, see the [README](./README.md) for installation instructions.

## Problem Statement

Consider the equation

$$
\begin{aligned}
    \tag{4.1}
    \frac{\partial^2}{\partial t^2}y(\boldsymbol x,t)
    = \nabla \cdot \left[c(\boldsymbol x,\boldsymbol{\mu})^2 \nabla y(\boldsymbol x,t)\right],
\end{aligned}
$$

defined for the two-dimensional spatial variable $x \in \Omega = [0, L]^2$ and the parameter vector $\boldsymbol{\mu}\in\mathbb{R}^{4}$, with homogeneous Dirichlet boundary conditions 

$$
    y(\boldsymbol x,t) = 0, \quad \boldsymbol x \in \partial \Omega
$$

and prescribed, parameter-independent initial conditions 

$$
    y(\boldsymbol{x},0) = \exp\left(-(x_1 - L/2)^2 - (x_2 - L/2)^2\right)\sin(x_1/2)\sin(x_2/2), \quad \frac{\partial}{\partial t} y(x,0) = 0,
$$ 

where $\boldsymbol{x} = [~x_1~~x_2~]^\mathsf{T}$. The wave speed $c(\boldsymbol x, \boldsymbol \mu)$ is piecewise constant over a partition of the  domain, $\Omega = \Omega_1\cup\Omega_2\cup\Omega_3\cup\Omega_4$ so that

$$
\begin{aligned}
    c(x,\boldsymbol{\mu})^2 = \begin{cases}
        \mu_1^2, & \boldsymbol{x}\in\Omega_1, \\
        \mu_2^2, & \boldsymbol{x}\in\Omega_2, \\
        \mu_3^2, & \boldsymbol{x}\in\Omega_3, \\
        \mu_4^2, & \boldsymbol{x}\in\Omega_4, \\
    \end{cases}
    \quad
    \boldsymbol{\mu} = \left[\begin{array}{c}
        \mu_1 \\ \mu_2 \\ \mu_3 \\ \mu_4
    \end{array}\right].
\end{aligned}
$$

It can be shown that $(4.1)$ has a Hamiltonian form in the canonical variables $q(x,t) := y(x,t)$ and $p(x,t) := \frac{\partial}{\partial t}y(x,t)$.

The class [`waveEq.WaveFEM2D`](./waveEq.py) discretizes $(4.1)$ with a [symplectic Hamiltonian mixed finite element method](https://www.sciencedirect.com/science/article/pii/S0045782521001808) using the [`ngsolve`](https://ngsolve.org/) library.
The resulting semi-discrete system has the form

$$
\begin{aligned}
    \tag{4.2}
    \left[\begin{array}{ c  c }
        \boldsymbol{M}_W & \boldsymbol{0} \\
        \boldsymbol{0} & \boldsymbol{M}_W
    \end{array}\right]
    \left[\begin{array}{c} {d\boldsymbol{q}}/{dt} \\ {d\boldsymbol{p}}/{dt} \end{array}\right]
    = \left[\begin{array}{ c  c }
    \boldsymbol{0} & \boldsymbol{I} \\
    -\boldsymbol{I} & \boldsymbol{0}
    \end{array}\right]
     \left[\begin{array}{ c  c }
        \boldsymbol{A}(\boldsymbol{\mu}) & \boldsymbol{0} \\
        \boldsymbol{0} & \boldsymbol{M}_W
    \end{array}\right] 
    \left[\begin{array}{c}
        \boldsymbol{q} \\
        \boldsymbol{p}
    \end{array}\right],
\end{aligned}
$$

where $\boldsymbol{M}_W \in \mathbb R^{N\times N}$ and $\boldsymbol A(\boldsymbol \mu) = \mu_1^{-2} \boldsymbol A_1 + \cdots + \mu_4^{-2} \boldsymbol A_4  \in \mathbb R^{N\times N}$. Note that $(4.2)$ has a simplified form with two blocks:

$$
\begin{aligned}
    \tag{4.3a}
    \boldsymbol M_W\frac{\text{d}\boldsymbol{q}}{\text{d}t}
    = \boldsymbol M_W\boldsymbol{p}(t),
\end{aligned}
$$

$$
\begin{aligned}
    \tag{4.3b}
    \boldsymbol{M}_W\frac{\text{d}\boldsymbol{p}}{\text{d}t}
    = - \boldsymbol{A}\boldsymbol({\mu})\boldsymbol{q}(t)
    = - (\mathbf{T}\boldsymbol{\mu'})\boldsymbol{q}(t)
\end{aligned}
$$

where $\boldsymbol \mu' = [\mu_1^{-2} \cdots \mu_4^{-2}]^\mathsf{T}$, and $\mathbf{T}\in\mathbb{R}^{N\times N\times 4}$ is a third-order tensor.

The Hamiltonian for the system $(4.3)$ is given by the expression
$$
\begin{aligned}
    \tag{4.4}
    H(\boldsymbol{q}, \boldsymbol{p}, \boldsymbol{\mu})
    = \frac{1}{2}\boldsymbol{p} ^\mathsf{T} \boldsymbol{M}_W \boldsymbol{p}
    + \frac{1}{2} \boldsymbol{q}^\mathsf{T}(\mathbf{T}\boldsymbol{\mu}')\boldsymbol{q}.
\end{aligned}
$$

This function should be preserved over time, i.e., $\frac{\textrm{d}}{\textrm{d}t}H(\boldsymbol{q}(t), \boldsymbol{p}(t), \boldsymbol{\mu}) = 0$ for all $\boldsymbol{\mu}$.

**Objectives**:
- Given samples of the solution to $(4.2)$-$(4.3)$ for various instances of $\boldsymbol{\mu}$, construct parametric reduced-order models (ROMs) for $(4.2)$-$(4.3)$ and infer the symmetric RHS tensor operator in $(4.3\text{b})$ using the algorithms described in the paper. The ROM can be solved for arbitrary choices of $\boldsymbol{\mu}$.
- Measure ROM accuracy and compare the results to classical intrusive Galerkin ROMs.
- Study the ROM accuracy as a function of the ROM size.
- Compare symmetric and non-symmetric OpInf ROMs.

## Training/Testing Data Generation

We start by constructing the finite element model $(4.2)$, also called the full-order model (FOM).

In [ ]:
t = np.linspace(0, 4 * np.pi, 501)
fom = wave.WaveFEM2D(L=2 * np.pi, h=0.4, orderW=2, orderV=2)
print(fom)

### Sample the Parameter Space

We randomly sample parameters $\boldsymbol{\mu}$, the entries of which are uniformly spaced in $(0.8, 2.4)$.
The samples are split into distinct training and testing sets.

In [ ]:
training_parameters, testing_parameters = fom.sample_parameters(
    low=0.8,
    high=2.4,
    num_samples=25,
    train_ratio=0.80,
    randseed=0,
)

print(f"{training_parameters.shape=}")
print(f"{testing_parameters.shape=}")

In [ ]:
# The parameters are four-dimensional. Plot the samples in several 2D spaces.
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
axes = axes.flatten()
param_indices = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]

for ax, ij in zip(axes, param_indices):
    i, j = ij
    ax.plot(
        training_parameters[:, i],
        training_parameters[:, j],
        "k*",
        label="training parameter values",
        markersize=8,
        markeredgewidth=0,
    )
    ax.plot(
        testing_parameters[:, i],
        testing_parameters[:, j],
        "C3.",
        label="testing parameter values",
        markersize=10,
        markeredgewidth=0,
    )
    ax.set_xlabel(rf"$\mu_{i+1}$")
    ax.set_ylabel(rf"$\mu_{j+1}$")

fig.tight_layout()
fig.subplots_adjust(bottom=0.2)
leg = axes[0].legend(
    ncol=2,
    loc="lower center",
    bbox_to_anchor=(0.5, 0),
    bbox_transform=fig.transFigure,
)
for line in leg.get_lines():
    line.set_markersize(20)

plt.show()

In [ ]:
training_Q, training_P = fom.solve_multi(training_parameters, t)
testing_Q, testing_P = fom.solve_multi(testing_parameters, t)

training_snapshots = np.concatenate((training_Q, training_P), axis=1)
testing_snapshots = np.concatenate((testing_Q, testing_P), axis=1)

print(f"{training_snapshots.shape=}")
print(f"{testing_snapshots.shape=}")

In [ ]:
# Visualize a single testing trajectory.
idx = 3
fom.plot(testing_Q[idx])

Next, we compute the Hamiltonian $(4.4)$ for this trajectory. The Hamiltonian should be constant along solution trajectories, so the standard deviation of the function values should be near machine precision.

In [ ]:
H = fom.Hamiltonian(testing_Q[idx], testing_P[idx], testing_parameters[idx])

print(f"Standard deviation of Hamiltonian values: {np.std(H):.6e}")

## Dimensionality Reduction

Now we need to compute a PSD basis matrix $\text{blockdiag}(\boldsymbol{U},\boldsymbol{U})\in\mathbb{R}^{N\times r}$ for approximating the FOM state as $\boldsymbol{q} \approx \boldsymbol{U}\hat{\boldsymbol{q}}$ and $\boldsymbol{p} \approx \boldsymbol{U}\hat{\boldsymbol{p}}$ where the full state is $\boldsymbol{y} = [~\boldsymbol{q}^\mathsf{T}~~\boldsymbol{p}^\mathsf{T}~]^{\mathsf{T}}$.
We use (weighted) proper orthogonal decomposition (POD), closely related to the SVD and PCA, to construct $\boldsymbol{U}$ such that $\boldsymbol{U}^{\mathsf{T}}\boldsymbol{M}_{W}\boldsymbol{U} = \boldsymbol{I}.$

To get started, extract the mass matrix $\boldsymbol{M}_{W}$.

In [ ]:
M = fom.MW.toarray()
print(f"Mass matrix: {type(M)=}, {M.shape=}")

Next, learn the basis by taking the (weighted) SVD of the matrix containing all training snapshots as columns.
The singular value decay gives a sense for how efficient the approximation is: a fast decay means that the state can be approximated well with only a few degrees of freedom.

In [ ]:
# Compute the PSD basis diag(U, U) such that U^T M U is the identity.
basis = models.PSDBasis(num_vectors=30, weights=M).fit(
    np.hstack(training_snapshots)
)
print(basis)

In [ ]:
# Plot the singular value decay.
fig, ax = plt.subplots(1, 1, figsize=(12, 4))
basis.pod.plot_svdval_decay(right=50, ax=ax)
fig.tight_layout()
plt.show()

## Reduced-order Models

With training data and a basis in hand, we are ready to construct reduced-order models (ROMs) for $(4.2)$-$(4.3)$.

### Construct ROMs with Intrusive Projection

Before constructing ROMs from data, we use classical Galerkin projection to construct a ROM consisting of the following two blocks:

$$
\begin{aligned}
    \frac{\text{d}\hat{\boldsymbol{q}}}{\text{d}t}
    = \hat{\boldsymbol{p}}(t)
\end{aligned}
$$

$$
\begin{aligned}
    \frac{\text{d}\hat{\boldsymbol{p}}}{\text{d}t}
    = -\boldsymbol{U}^{\mathsf{T}}{\boldsymbol{A}(\boldsymbol{\mu})}\boldsymbol{U}\hat{\boldsymbol{q}}(t)
    = - (\mu_1^{-2}\boldsymbol{U}^{\mathsf{T}}\boldsymbol{A}_1\boldsymbol{U} + \cdots + \mu_4^{-2}\boldsymbol{U}^{\mathsf{T}}\boldsymbol{A}_4\boldsymbol{U})\hat{\boldsymbol{q}}(t)
\end{aligned}
$$

In [ ]:
basis.set_dimension(num_vectors=18)
training_Q_galerkin, training_P_galerkin = fom.solve_multiROM(
    basis.pod, training_parameters, t
)
testing_Q_galerkin, testing_P_galerkin = fom.solve_multiROM(
    basis.pod, testing_parameters, t
)

The following blocks compute the relative errors for the training and testing datasets. These errors are evaluated using a space-time $L^2(\Omega) \times [t_0, t_f]$ norm on the position $\boldsymbol{q}(t)$; see [`utils.solution_error()`](./utils.py) for implementation details.

In [ ]:
# Error over the training set.
galerkin_training_error = utils.solution_error(
    training_Q, training_Q_galerkin, M=basis.pod.weights
)

print(f"Training error: {galerkin_training_error:.4%}")

In [ ]:
# Error over the testing set.
galerkin_testing_error = utils.solution_error(
    testing_Q, testing_Q_galerkin, M=basis.pod.weights
)

print(f"Testing error: {galerkin_testing_error:.4%}")

We also verify that the Galerkin ROM preserves the reduced Hamiltonian, which is given by

$$
\begin{aligned}
    \tag{3.5}
    H(\hat{\boldsymbol q}, \hat{\boldsymbol p}, \boldsymbol \mu)
    = \frac{1}{2}\hat{\boldsymbol p} ^\mathsf{T} \hat{\boldsymbol p}
    + \frac{1}{2} \hat{\boldsymbol q}^\mathsf{T}\hat{\boldsymbol A}(\boldsymbol \mu)\hat{\boldsymbol q},
\end{aligned}
$$

where $\hat{\boldsymbol A}(\boldsymbol \mu) = \mu_1^{-2}\hat{\boldsymbol{A}}_1 + \cdots + \mu_4^{-2}\hat{\boldsymbol{A}}_4$.

In [ ]:
H_galerkin = fom.reduced_Hamiltonian(
    basis.pod,
    testing_Q_galerkin[idx],
    testing_P_galerkin[idx],
    testing_parameters[idx],
)

print(
    "Standard deviation of reduced Hamiltonian values:",
    f"{np.std(H_galerkin):.6e}",
)

# Used in WaveBlowup.ipynb.
utils.savenpy("wave2D_reducedHamiltonian_intrusive.npy", H_galerkin)

### Construct ROMs with Operator Inference

This section applies tensor parametric operator inference to learn ROMs from data without intrusive projection. We learn ROMs of the with the same block form as the FOM,

$$
\begin{gathered}
    \frac{\text{d}\hat{\boldsymbol{q}}}{\text{d}t}
    = \bar{\boldsymbol{A}}\hat{\boldsymbol{p}}(t),
    \qquad
    \frac{\text{d}\hat{\boldsymbol{p}}}{\text{d}t}
    = -(\bar{\mathbf{T}} \boldsymbol{\mu}')\hat{\boldsymbol{q}}(t),
\end{gathered}
$$

and infer the operators using two strategies:

- Structure-preserving Hamiltonian ROMs (H-OpInf) learn the reduced tensor operator $\bar{\mathbf{T}}$ using Algorithm 3.1 and the reduced matrix operator $\bar{\boldsymbol{A}}$ using standard Hamiltonian OpInf.
- Structure-agnostic ROMs (OpInf) learn $\bar{\mathbf{T}}$ using  Algorithm 2.1 and $\bar{\boldsymbol{A}}$ using standard OpInf.

The time derivatives of the (reduced) state data are estimated using a finite difference scheme.

In [ ]:
ddter = opinf.ddt.UniformFiniteDifferencer(t, "ord4")

opinf_rom_sym = opinf.ParametricROM(
    basis=basis,
    ddt_estimator=ddter,
    model=models.BlockHamiltonianTensorModel(
        fom.parameter_dimension,
        symmetric=True,
    ),
).fit(training_parameters, training_snapshots, fit_basis=False)

opinf_rom_nosym = opinf.ParametricROM(
    basis=opinf_rom_sym.basis,
    ddt_estimator=ddter,
    model=models.BlockHamiltonianTensorModel(
        fom.parameter_dimension,
        symmetric=False,  # No symmetry constraint
    ),
).fit(training_parameters, training_snapshots, fit_basis=False)

We evaluate each ROM over the training and testing parameter sets and check their Hamiltonians.

In [ ]:
def predict_multi(self, muarr, state0, t):
    """Solve the problem for multiple parameter vectors."""
    Q_list, P_list = [], []
    state0 = state0.reshape((-1))
    for mu in muarr:
        Y = self.predict(mu, state0, t)

        Q_list.append(Y[: len(Y) // 2, :])
        P_list.append(Y[len(Y) // 2 :, :])

    return np.array(Q_list), np.array(P_list)

#### Symmetric ROM from Hamiltonian Operator Inference

In [ ]:
# Evaluate the symmetric OpInf ROM over the training and testing sets.
y0 = training_snapshots[0][:, 0]
training_Q_symOpInf, training_P_symOpInf = predict_multi(
    opinf_rom_sym, training_parameters, y0, t
)
testing_Q_symOpInf, testing_P_symOpInf = predict_multi(
    opinf_rom_sym, testing_parameters, y0, t
)

In [ ]:
# Error over the training set.
symOpInf_training_error = utils.solution_error(
    training_Q, training_Q_symOpInf, M=basis.pod.weights
)

print(f"Training error: {symOpInf_training_error:.4%}")

In [ ]:
# Error over the testing set.
symOpInf_testing_error = utils.solution_error(
    testing_Q, testing_Q_symOpInf, M=basis.pod.weights
)

print(f"Testing error: {symOpInf_testing_error:.4%}")

In [ ]:
# Reduced Hamiltonian at a single testing parameter.
Y = np.concatenate((testing_Q_symOpInf, testing_P_symOpInf), axis=1)
H_symOpInf = opinf_rom_sym.model.Hamiltonian(
    opinf_rom_sym.encode(Y[idx]),
    testing_parameters[idx],
)

print(
    "Standard deviation of reduced Hamiltonian values:",
    f"{np.std(H_symOpInf):.6e}",
)

# Used in WaveBlowup.ipynb.
utils.savenpy("wave2D_reducedHamiltonian_H-OpInf.npy", H_symOpInf)

#### Non-symmetric ROM from Standard OpInf

In [ ]:
# Evaluate the symmetric OpInf ROM over the training and testing sets.
training_Q_nosymOpInf, training_P_nosymOpInf = predict_multi(
    opinf_rom_nosym, training_parameters, y0, t
)
testing_Q_nosymOpInf, testing_P_nosymOpInf = predict_multi(
    opinf_rom_nosym, testing_parameters, y0, t
)

In [ ]:
# Error over the training set.
nosymOpInf_training_error = utils.solution_error(
    training_Q, training_Q_nosymOpInf, M=basis.pod.weights
)

print(f"Training error: {nosymOpInf_training_error:.4%}")

In [ ]:
# Error over the testing set.
nosymOpInf_testing_error = utils.solution_error(
    testing_Q, testing_Q_nosymOpInf, M=basis.pod.weights
)

print(f"Testing error: {nosymOpInf_testing_error:.4%}")

In [ ]:
# Reduced Hamiltonian at a single testing parameter.
Y = np.concatenate((testing_Q_nosymOpInf, testing_P_nosymOpInf), axis=1)
H_nosymOpInf = opinf_rom_nosym.model.Hamiltonian(
    opinf_rom_nosym.encode(Y[idx]),
    testing_parameters[idx],
)

print(
    "Standard deviation of reduced Hamiltonian values:",
    f"{np.std(H_nosymOpInf):.6e}",
)

# Used in WaveBlowup.ipynb.
utils.savenpy("wave2D_reducedHamiltonian_OpInf.npy", H_nosymOpInf)

The standard non-symmetric OpInf ROM outperforms the intrusive Galerkin ROM and the symmetric OpInf ROM in an $L^2$ sense, but the reduced Hamiltonian is no longer constant in time. This can lead to numerical blowup, which is demonstrated in more detail in [WaveBlowup.ipynb](./WaveBlowup.ipynb).

In [ ]:
# Plot the Hamiltonian values.
fig, ax = plt.subplots(1, 1, figsize=(12, 4))

ax.semilogy(t, np.abs(H_galerkin - H_galerkin[0]), "C2-", label="Intrusive")
ax.semilogy(t, np.abs(H_symOpInf - H_symOpInf[0]), "C0-.", label="H-OpInf")
ax.semilogy(t, np.abs(H_nosymOpInf - H_nosymOpInf[0]), "C1:", label="OpInf")
ax.set_ylabel("|")

fig.subplots_adjust(right=0.85)
ax.legend(
    loc="right",
    bbox_to_anchor=(1, 0.5),
    bbox_transform=fig.transFigure,
)

plt.show()

### Sensitivity to Basis Size

We now train ROMs of different sizes, meaning the we change the number of columns in the basis matrix $\boldsymbol{U}$, and compute errors over the training and testing sets.

In [ ]:
def rom_error(rom, testing: bool = False, total: bool = False):
    """Calculate ROM errors over the training or testing sets.

    Parameters
    ----------
    rom : opinf.ParametricROM
        Reduced-order model to test.
    testing : bool
        If ``False`` (default), evaluate the ROM on all training parameters.
        If ``True``, evaluate the ROM on all testing parameters.
    total : bool
        If ``False`` (default), return the errors for each parameter.
        If ``True``, return the total error over the parameter set.

    Returns
    -------
    error(s) : ndarray or float
        Relative ROM errors, for each parameter (``total=False``)
        or for all parameters together (``total=True``).
    """
    args = (training_parameters, training_snapshots)
    if testing:
        args = (testing_parameters, testing_snapshots)
    W = rom.basis.pod.weights

    nx = args[1][0].shape[0] // 2
    errors, rom_solutions = [], []
    for mu, Y in zip(*args):
        # Solve the ROM at the given parameter value.
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            Yrom = rom.predict(mu, Y[:, 0], t=t)
        # Check that the integration succeeded.
        if Yrom.shape != Y.shape or np.any(np.isnan(Yrom)):
            print(f"integration failed at {mu=}")
            return np.nan
        Y, Yrom = Y[:nx], Yrom[:nx]

        # Record the results.
        if total:
            rom_solutions.append(Yrom)
        else:
            errors.append(utils.solution_error(Y, Yrom, M=W))

    if total:
        Y_all = args[1]
        Y_all = np.array([Y[:nx] for Y in Y_all])
        return utils.solution_error(Y_all, np.array(rom_solutions), M=W)

    return np.array(errors)

In [ ]:
results = collections.defaultdict(list)
# rom_classes = [models.HamiltonianTensorModel, models.NormalTensorModel]
labels = ["h-opinf", "opinf"]

basis = opinf_rom_sym.basis

print("r =", end="")
for r in (rs := list(range(1, 40))):
    print(f" {r}", end="")
    basis.set_dimension(num_vectors=r)

    # Projection errors.
    results["train-project"].append(
        utils.projection_error(training_Q, basis.pod)
    )

    results["test-project"].append(
        utils.projection_error(testing_Q, basis.pod)
    )

    # Intrusive ROM errors
    results["train-intrusive"].append(
        utils.solution_error(
            training_Q,
            fom.solve_multiROM(basis.pod, training_parameters, t)[0],
            M=basis.pod.weights,
        )
    )

    results["test-intrusive"].append(
        utils.solution_error(
            testing_Q,
            fom.solve_multiROM(basis.pod, testing_parameters, t)[0],
            M=basis.pod.weights,
        )
    )

    # Instantiate / train ROMs.
    roms = [
        opinf.ParametricROM(
            basis=basis,
            ddt_estimator=ddter,
            model=models.BlockHamiltonianTensorModel(
                fom.parameter_dimension,
                symmetric=True,
            ),
        ).fit(training_parameters, training_snapshots, fit_basis=False),
        opinf.ParametricROM(
            basis=basis,
            ddt_estimator=ddter,
            model=models.BlockHamiltonianTensorModel(
                fom.parameter_dimension,
                symmetric=False,
            ),
        ).fit(training_parameters, training_snapshots, fit_basis=False),
    ]

    # Evaluate ROMs.
    for rom, label in zip(roms, labels):
        results[f"train-{label}"].append(
            rom_error(rom, testing=False, total=True)
        )
        results[f"test-{label}"].append(
            rom_error(rom, testing=True, total=True)
        )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)

key_style_label = (
    ("project", "-k", "Projection"),
    ("intrusive", "C2:o", "Intrusive"),
    ("h-opinf", "C0--d", "H-OpInf"),
    ("opinf", "C1:s", "OpInf"),
)

for ax, which in zip(axes, ["train", "test"]):
    for key, style, label in key_style_label:
        ax.semilogy(
            rs,
            results[f"{which}-{key}"],
            style,
            label=label,
            lw=1.5,
            markersize=8,
        )

    ax.set_xlabel(r"Reduced dimension $r$", fontsize=28)
    ax.tick_params(labelsize=28)
    ax.set_xlim(0, rs[-1] + 0.5)
    ax.set_yticks(
        [1e-2, 5e-2, 2e-1],
        [r"1\%", r"5\%", r"20\%"],
    )
    ax.grid(which="major", axis="y", lw=0.5, color="gray")
    ax.annotate(
        f"{'Training' if which=="train" else 'Testing'} set",
        xy=(0.95, 0.875),
        xycoords="axes fraction",
        ha="right",
        fontsize=26,
    )
axes[0].set_ylabel("Relative error", fontsize=28)

fig.tight_layout()
fig.subplots_adjust(bottom=0.35)
leg = axes[0].legend(
    ncol=len(key_style_label),
    loc="lower center",
    bbox_to_anchor=(0.5, 0),
    bbox_transform=fig.transFigure,
    framealpha=0,
    fontsize=30,
)
for line in leg.get_lines():
    line.set_markersize(12)
    line.set_linewidth(4)

plt.show()

## Contour plots

This section contains code that generates the VTK files needed for reproducing Figure 4.11 in the paper.

In [ ]:
basis.set_dimension(num_vectors=18)

# Build and train opinf ROMs
ddter = opinf.ddt.UniformFiniteDifferencer(t, "ord4")
opinfROM = opinf.ParametricROM(
    basis=basis,
    ddt_estimator=ddter,
    model=models.BlockHamiltonianTensorModel(
        fom.parameter_dimension,
        symmetric=True,
    ),
).fit(training_parameters, training_snapshots, fit_basis=False)

opinfROM_V = opinf.ParametricROM(
    basis=opinfROM.basis,
    ddt_estimator=ddter,
    model=models.BlockHamiltonianTensorModel(
        fom.parameter_dimension,
        symmetric=False,  # No symmetry constraint
    ),
).fit(training_parameters, training_snapshots, fit_basis=False)

In [ ]:
idx = 3
state0 = testing_snapshots[idx][:, 0]
print(testing_parameters[idx])

In [ ]:
# Obtain intrusive ROM solution
QI, PI = fom.solveROM(basis.pod, testing_parameters[idx], t)
print(utils.solution_error(testing_Q[idx], QI, M=basis.pod.weights))

In [ ]:
# Obtain H-OpInf ROM solution
YHO = opinfROM.predict(testing_parameters[idx], state0, t)
QHO = np.vsplit(YHO, 2)[0]
print(utils.solution_error(testing_Q[idx], QHO, M=basis.pod.weights))

In [ ]:
# Obtain OpInf ROM solution
YO = opinfROM_V.predict(testing_parameters[idx], state0, t)
QO = np.vsplit(YO, 2)[0]
print(utils.solution_error(testing_Q[idx], QO, M=basis.pod.weights))

In [ ]:
# Save FOM solutions
fom.save_VTK(testing_Q[idx][:, 0], "FOM_I")
fom.save_VTK(testing_Q[idx][:, t.shape[0] // 2], "FOM_M")
fom.save_VTK(testing_Q[idx][:, -1], "FOM_F")

# Save ROM solutions at final time
fom.save_VTK(QI[:, -1], "intrusive")
fom.save_VTK(QHO[:, -1], "hopinf")
fom.save_VTK(QO[:, -1], "opinf")